# 01 — Exploratory Data Analysis: Soil Moisture Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/data'
FIG_DIR = '/content/drive/MyDrive/BSE405_SoilMoisture/figures'

import os
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(f'{DATA_DIR}/updated_data.csv', parse_dates=['time'])
print(f"Shape: {df.shape}")
df.head()

## Summary Statistics

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Summary Statistics ===")
df.describe()

## Missing Value Analysis

In [ ]:
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")
print(f"Rows with any missing: {df.isnull().any(axis=1).sum()}")

## Feature Distributions

In [ ]:
numeric_cols = ['clay_content', 'sand_content', 'silt_content', 'sm_aux', 'sm_tgt']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')
plt.suptitle('Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlation Heatmap

In [ ]:
corr_cols = ['latitude', 'longitude', 'clay_content', 'sand_content', 'silt_content', 'sm_aux', 'sm_tgt']
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Spatial Distribution

In [ ]:
grid = df.groupby(['latitude', 'longitude']).size().reset_index(name='count')

plt.figure(figsize=(10, 8))
scatter = plt.scatter(grid['longitude'], grid['latitude'], c=grid['count'],
                      cmap='YlOrRd', s=20, edgecolors='gray', linewidth=0.3)
plt.colorbar(scatter, label='Number of Observations')
plt.xlabel('Longitude (\u00b0)')
plt.ylabel('Latitude (\u00b0)')
plt.title('Spatial Distribution of Observations (Germany)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/spatial_map.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Unique grid points: {len(grid)}")
print(f"Lat range: {df['latitude'].min():.2f} to {df['latitude'].max():.2f}")
print(f"Lon range: {df['longitude'].min():.2f} to {df['longitude'].max():.2f}")

## Temporal Coverage

In [ ]:
daily_count = df.groupby('time').size()

plt.figure(figsize=(12, 4))
plt.plot(daily_count.index, daily_count.values, linewidth=0.8)
plt.xlabel('Date')
plt.ylabel('Number of Observations')
plt.title('Temporal Coverage (2013)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/temporal_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Date range: {df['time'].min()} to {df['time'].max()}")
print(f"Unique dates: {df['time'].nunique()}")

## Auxiliary vs Target Soil Moisture

In [ ]:
sample = df.sample(n=5000, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(sample['sm_aux'], sample['sm_tgt'], alpha=0.3, s=5)
plt.xlabel('sm_aux (SMOS-ASCAT, m\u00b3/m\u00b3)')
plt.ylabel('sm_tgt (AMSR, m\u00b3/m\u00b3)')
plt.title('Auxiliary vs Target Soil Moisture')
plt.plot([0, 1], [0, 1], 'r--', label='1:1 line')
plt.legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/aux_vs_tgt_scatter.png', dpi=150, bbox_inches='tight')
plt.show()